In [1]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime
from functools import partial

# local imports
import sys
# sys.path.append('../../../')
sys.path.append('/exp/sbnd/app/users/munjung/xsec/freeze/cafpyana') # absolute path for running on EAF
from pyanalib.split_df_helpers import *
from analysis_village.numucc_1p0pi.variable_configs import VariableConfig
from analysis_village.numucc_1p0pi.utils import *
from analysis_village.numucc_1p0pi.files_config import *
from analysis_village.numucc_1p0pi.constants import *
from analysis_village.unfolding.wienersvd import *
from pyanalib.covariance import *

import matplotlib.pyplot as plt 

import warnings
from pandas.errors import PerformanceWarning
warnings.filterwarnings("ignore", category=PerformanceWarning)

In [4]:
from pyanalib.split_df_helpers_new import *

filename = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_024530__sel_mup-wgts_genie_CCQE/merged_perTPC/2026_05_11_024530__sel_mup-wgts_genie_CCQE_merged_0000.df"
df_dict = load_dfs(filename, ["mcnu"])
mcnudf = df_dict['mcnu']

In [5]:
mcnudf

mc                                  \
                                        E    baseline      time  bjorkenX   
                                                                            
                                                                            
__ntuple entry rec.mc.nu..index                                             
2        0     0                 0.717379  100.464722  1.690284  0.355664   
               1                 1.131408   89.671593  0.973344  0.511831   
               2                 0.715906  117.863945  1.731260  1.609359   
         1     0                 1.093659  126.786728  1.794368  0.796093   
         2     0                 0.916819  125.671303  1.146628  1.763079   
...                                   ...         ...       ...       ...   
18700    16    1                 1.073225  117.773132  0.961847  0.151686   
               2                 0.470550   84.103317  0.489636  0.645273   
         17    0                 0.573617  125.233421  1.110081  0.622133   
               1                 0.475275   83.447273  1.034822  0.975427   
         18    0                 1.452881   90.107330  1.717828  0.390581   

                                                                             \
                                inelasticityY        Q2         w  momentum   
                                                                          x   
                                                                              
__ntuple entry rec.mc.nu..index                                               
2        0     0                     0.463871  0.222251  1.133229  0.016551   
               1                     0.735316  0.799608  1.282268  0.004530   
               2                     0.174131  0.376741  0.859605  0.009750   
         1     0                     0.666146  1.089113  1.077278 -0.005441   
         2     0                     0.197899  0.600698  0.788404 -0.008641   
...                                       ...       ...       ...       ...   
18700    16    1                     0.432968  0.132358  1.273495 -0.037107   
               2                     0.513783  0.292946  1.021083  0.007020   
         17    0                     0.549256  0.368076  1.051251  0.007643   
               1                     0.310943  0.270694  0.942543  0.011949   
         18    0                     0.236402  0.251913  1.128993  0.000746   

                                                     ...  \
                                                     ...   
                                        y         z  ...   
                                                     ...   
__ntuple entry rec.mc.nu..index                      ...   
2        0     0                 0.011597  0.717095  ...   
               1                -0.004936  1.131388  ...   
               2                -0.012734  0.715726  ...   
         1     0                 0.017826  1.093501  ...   
         2     0                 0.005888  0.916759  ...   
...                                   ...       ...  ...   
18700    16    1                 0.001515  1.072582  ...   
               2                -0.005694  0.470463  ...   
         17    0                -0.008592  0.573502  ...   
               1                 0.034060  0.473903  ...   
         18    0                 0.008947  1.452854  ...   

                                                                           \
                                GENIEReWeight_SBN_v1_multisim_CoulombCCQE   
                                                                  univ_98   
                                                                            
__ntuple entry rec.mc.nu..index                                             
2        0     0                                                 1.000000   
               1                                                 1.000000   
               2                         

In [ ]:
filename = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_030547__sel_mup-wgts_genie_RES/sel_mup-wgts_genie_RES_322.df"
df_dict = load_dfs(filename, ["evt"])
df_dict['evt']

In [ ]:
from pyanalib.split_df_helpers_new import *

df_dir = "/pnfs/sbnd/scratch/users/munjung/cafpyana_out/dfs/2026_05_11_024530__sel_mup-wgts_genie_CCQE"
keys2load_data = ['hdr', 'evt', 'mcnu']
df_data = dfs_from_dir(search_dir=df_dir, filename_str="sel_mup", keys2load=keys2load_data, n_max_concat=100)
mc_evt_df = df_data['evt']
mc_hdr_df = df_data['hdr']
mc_nu_df = df_data['mcnu']

In [ ]:
mc_nu_df.columns = pd.MultiIndex.from_tuples([tuple(["mc"] + list(c)) for c in mc_nu_df.columns])     # match # of column levels
mc_nu_df.loc[:,'topo_categ'] = get_topo_category(mc_nu_df)
mc_nu_df.loc[:,'genie_categ'] = get_genie_category(mc_nu_df)

mc_evt_df.loc[:,'topo_categ'] = get_topo_category(mc_evt_df)
mc_evt_df.loc[:,'genie_categ'] = get_genie_category(mc_evt_df)

from pyanalib.variable_calculator import *

tki_var_names = ["del_alpha", "del_phi", "del_Tp", "del_p", "del_Tp_x", "del_Tp_y"]

slc_mudf = mc_evt_df.mu.pfp.trk.truth.p
slc_pdf = mc_evt_df.p.pfp.trk.truth.p
slc_P_mu_col = pad_column_name(("totp",), slc_mudf)
slc_P_p_col = pad_column_name(("totp",), slc_pdf)
tki_reco = get_cc1p0pi_tki(slc_mudf, slc_pdf, slc_P_mu_col, slc_P_p_col)
for var_name in tki_var_names:
    mc_evt_df = multicol_add(mc_evt_df, tki_reco[var_name].rename("mc_" + var_name))

mc_mudf = mc_nu_df.mc.mu
mc_pdf = mc_nu_df.mc.p
mc_P_mu_col = pad_column_name(("totp",), mc_mudf)
mc_P_p_col = pad_column_name(("totp",), mc_pdf)
tki_mc = get_cc1p0pi_tki(mc_mudf, mc_pdf, mc_P_mu_col, mc_P_p_col)
for var_name in tki_var_names:
        mc_nu_df = multicol_add(mc_nu_df, tki_mc[var_name].rename("{}".format(var_name)))